# Task 7 — Smoke test TimesNet → SSA

Chạy toàn bộ notebook bằng kernel Python tương thích với `Task03_Environment_Setup/` và `requirements.txt`. Notebook **không tự cài package hoặc sửa môi trường**. `smoke_pipeline.py` kiểm tra import NumPy, pandas, PyArrow và PyTorch; dùng Parquet Task 6 nếu hash khớp, nếu không có thì nạp split chính thức `Dingdong-Inc/FreshRetailNet-50K`.

Pipeline chọn 100 cặp theo seed 2024, lưu train/eval smoke riêng, và truyền cùng đường dẫn Parquet train tường minh cho `load_data` lẫn `_demand_recovery`. SSA raw và SSA TimesNet nhận cùng eval 700 dòng; eval không tham gia huấn luyện. TimesNet smoke dùng 1 epoch, batch size 128, seed baseline 1024. Dữ liệu, Parquet kết quả, CSV và log bị `.gitignore` loại khỏi Git theo Contribution Guide.

Nếu môi trường không đủ điều kiện, các kết quả được để `not_run` và không tạo Parquet giả. Không dùng số smoke trong Bảng 3.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Platform: {platform.platform()}")
for module_name in ("numpy", "pandas", "pyarrow", "torch", "datasets"):
    try:
        available = importlib.util.find_spec(module_name) is not None
    except (ImportError, ValueError):
        available = False
    print(f"{module_name}: {'available' if available else 'missing'}")
print("The next cell performs real imports and records any native/import failure.")

## Chạy pipeline

Tạo lại output từ đầu bằng cách chạy cell dưới đây. Nếu một bước lỗi, traceback và trạng thái sẽ được ghi trong `run_info.md` và log tương ứng. Không ghi đè một file `demand_timesnet_smoke.parquet` đã có.

In [ ]:
import runpy

search_roots = (Path.cwd(), *Path.cwd().parents)
task_dir = next(
    (root / "Task07_Smoke_Test" for root in search_roots
     if (root / "Task07_Smoke_Test" / "smoke_pipeline.py").is_file()),
    None,
)
if task_dir is None and (Path.cwd() / "smoke_pipeline.py").is_file():
    task_dir = Path.cwd()
if task_dir is None:
    raise FileNotFoundError("Could not locate Task07_Smoke_Test/smoke_pipeline.py")

pipeline_path = task_dir / "smoke_pipeline.py"
try:
    runpy.run_path(str(pipeline_path), run_name="__main__")
except SystemExit as exc:
    print(f"Pipeline exit code: {exc.code}; inspect run_info.md and smoke/*.log.")

## Kiểm tra bàn giao

Cell này chỉ xác nhận hình dạng đầu ra khi các stage thực sự hoàn tất; `not_run` không được tính là kết quả.

In [ ]:
import csv

pairs_path = task_dir / "smoke_pairs.csv"
metrics_path = task_dir / "smoke_metrics.csv"
with pairs_path.open(encoding="utf-8-sig", newline="") as stream:
    pairs = list(csv.DictReader(stream))
with metrics_path.open(encoding="utf-8-sig", newline="") as stream:
    metrics = list(csv.DictReader(stream))

print(f"Selected pair rows: {len(pairs)}")
print(f"Metric rows: {len(metrics)}")
print(metrics)
overall = [row for row in metrics if row["group"] == "Overall"]
other_statuses_valid = all(
    row["status"] in ("completed", "no_valid_rows")
    for row in metrics if row["group"] != "Overall"
)
if (len(pairs) == 100 and len(overall) == 2
        and all(row["status"] == "completed" for row in overall)
        and other_statuses_valid):
    assert (task_dir / "demand_timesnet_smoke.parquet").is_file()
    assert all(row["wape_pct"] and row["wpe_pct"] for row in overall)
    print("Smoke artifacts pass the final presence and metric checks.")
else:
    print("No successful smoke result is claimed; see run_info.md for not_run/failed details.")